In [1]:
import dspy
import pandas as pd
from tqdm import tqdm
from dspy.teleprompt import BootstrapFewShot
from rich import print 
from dspy.retrieve.chromadb_rm import ChromadbRM

from dspy.datasets import HotPotQA
from dspy.evaluate.evaluate import Evaluate
import random
from dspy.retrieve.faiss_rm import FaissRM
from langchain_community.llms import Ollama
# From LangChain, import standard modules for prompting.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils import embedding_functions
import chromadb

In [2]:
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

In [3]:
example1 = """
6 hours\nAssociate Actuary - SPA Rx\nCincinnati, OH 45217\n**Description** The Associate Actuary, Pricing is responsible for setting pricing assumptions, submitting bids, filing and gaining approval of premium rates and rate certifications with regulatory agencies. Supports implementation of rates, new plans and benefit changes. Provides guidance to Product Development on new product/benefit cost impacts. Develops and maintains pricing tools that price standard and custom benefits. The Associate Actuary, Pricing work assignments involve moderately complex to complex issues where the analysis of situations or data requires an in-depth evaluation of variable factors. **Responsibilities** _This a remote nationwide position_ The Associate Actuary, Pricing establishes market level financial metrics to align with segment profitability goals, analyzes market level results and projections and develops recommended pricing actions to address gaps to targeted metrics. Leverages market level projections and experience data tools to research root cause and capture insights. Researches and understands competitors in marketplace and collaborates with sales and other partners supporting the markets to develop strategies for profitable membership growth. Accountable for actuarial certifications on rate filings, including attesting to compliance with state and federal rating and benefit regulations. Begins to influence department's strategy. Makes decisions on moderately complex to complex issues regarding technical approach for project components, and work is performed without direction. Exercises considerable latitude in determining objectives and approaches to assignments. **Required Qualifications** + Bachelor's Degree + Associate of Society of Actuaries (ASA) designation + Meets eligibility requirements for Humana's Actuarial Professional Development Program (APDP) + MAAA + Strong communication + Must be passionate about contributing to an organization focused on continuously improving consumer experiences **Our Hiring Process** As part of our hiring process for this opportunity, we may contact you via text message and email to gather more information using a software platform called Modern Hire. Modern Hire Text, Scheduling and Video technologies allow you to interact with us at the time and location most convenient for you. If you are selected to move forward from your application prescreen, you may receive correspondence inviting you to participate in a pre-recorded Voice, Text Messaging and/or Video interview. Your recorded interview will be reviewed and you will subsequently be informed if you will be moving forward to next round of interviews If you have additional questions regarding this role posting and are an Internal Candidate, please send them to the Ask A Recruiter persona by visiting go/Buzz and searching Ask A Recruiter! Please be sure to provide the requisition number so we may be able to research your request quicker. **Alert:** Humana values personal identity protection. Please be aware that applicants selected for leader review may be asked to provide a social security number, if it is not already on file. When required, an email will be sent from Humana@myworkday.com with instructions to add the information into the application at Humana's secure website. **_Humana is more than an equal opportunity employer, Humana's dedication to promoting diversity, multiculturalism, and inclusion is at the heart of what we do in all of our Humana roles. Diversity is more than a commitment to us, it is the foundation of what we do. We are fully focused on diversity of race, gender, sexual orientation, religion, ethnicity, national origin and all of the other fascinating characteristics that make us each uniquely wonderful._** \#LI-Remote **Scheduled Weekly Hours** 40 Humana complies with all applicable federal civil rights laws and does not discriminate on the basis of race, color, national origin, age, disability, sex, sexual orientation, gender identity or religion. We also provide free language interpreter services. See our https://www.humana.com/legal/accessibility-resources?source=Humana_Website.
"""

In [4]:
texts = text_splitter.create_documents([example1])
ids = [str(x+1) for x,_ in enumerate(texts)]
docs = [doc.page_content for doc in texts]

In [5]:
chroma_client = client = chromadb.PersistentClient(path="./postings")
default_ef = embedding_functions.DefaultEmbeddingFunction()
collection = chroma_client.get_or_create_collection(name="postings", embedding_function=default_ef)

In [6]:
collection.add(
    documents=docs,
    ids=ids
)

rm = ChromadbRM(collection_name='postings', persist_directory="./postings", embedding_function=default_ef)

In [7]:
llm = dspy.OllamaLocal(model='phi3:14b')
dspy.settings.configure(lm=llm, rm=rm)

In [11]:
class RAG(dspy.Module):
    def __init__(self, num_passages=5):
        super().__init__()

        self.retrieve = dspy.Retrieve(k=num_passages)
        self.generate_answer = dspy.ChainOfThought("context, question -> answer", max_tokens = 1000)

    def forward(self, question):
        context = self.retrieve(question).passages
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

In [12]:
uncompiled_rag = RAG()

response = uncompiled_rag("Is there education requirements or certifications for this position?")

print(response.answer)

Based on the given context, it appears that there are certain educational requirements and certifications needed 
for the Associate Actuary - SPA Rx role at Humana. The job description mentions "Program (APDP)" which stands for 
American Academy of Actuarial & Financial Analysts' Program in Predictive Modeling. This suggests a need for 
knowledge or experience in predictive modeling, possibly through this specific program.

Additionally, the mention of "MAAA" indicates that candidates should have their Associate of the Society of 
Actuaries (ASA) designation or be eligible to sit for it. The ASA is a professional certification for actuaries in 
North America and requires

In [13]:
response = uncompiled_rag("Is there required years of experience for this role or specific job titles or skills needed for this position?")
print(response.answer)

Based on the provided information about the Associate Actuary - SPA Rx role at Humana, there are no explicit years 
of experience mentioned as a requirement. However, it is common for actuarial roles to require relevant education 
and certifications such as passing exams from professional bodies like the Society of Actuaries or Casualty 
Actuarial Society.

To determine if specific job titles or skills are needed for this position, we can follow these steps:

1. Review the provided information about the role again to look for any clues regarding required experience or 
qualifications. In this case, there is no mention of a minimum number of years of experience or previous job titles
that must be held by applicants.

In [14]:
response = uncompiled_rag("Where is this job located and is it remote, hybrid or onsite?")
print(response.answer)

Context: 
[1] «Program (PA) Rx
Cincinnati, OH 4521^3»
[2] «**Description** The Associate Actuary, Pricing is responsible for setting pricing assumptions, submitting 
bids, filing and gaining approval of premium rates and rate certifications with regulatory agencies. Supports 
implementation of rates, new plans and benefit changes. Provides guidance to Product Development on new 
product/benefit cost impacts. Develops and maintains pricing tools that price standard and custom benefits. The 
Associate Actuary, Pricing work assignments involve moderately complex to complex issues where the analysis of 
situations or

In [15]:
class GenerateAnswer(dspy.Signature):
    """Answer questions with short factoid answers."""

    context = dspy.InputField(desc="may contain relevant facts")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="often between 1 and 5 words")

In [30]:
class QUESTIONANSWER(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question,context):
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

In [31]:
uncompiled_fs = fs()


In [32]:
pred = uncompiled_fs("Where is this job located and is it remote, hybrid or onsite?",example1)

In [33]:
print(pred.answer)

This job opportunity is remote, meaning it does not have a specific location within Louisiana (#LI-Remote). The 
role allows for flexibility in terms of work arrangements, as there are no details about hybrid or onsite options 
mentioned in the description. You can perform this job from any location with a stable internet connection.

In [34]:
pred = uncompiled_fs("Is there required years of experience for this role or specific job titles or skills needed for this position?",example1)

In [35]:
print(pred.answer)

To determine whether there is a required number of years of experience, specific job titles, or skills needed for 
the Humana position you mentioned, please follow these steps:

1. Review the job posting on the Humana website (https://www.humana.com/careers/) or Modern Hire platform where it 
was posted. Look for sections that mention "Qualifications," "Requirements," or "Skills." These sections typically 
outline any necessary experience, skills, and/or certifications required for the position.

2. If you are unable to find this information in the job posting itself, visit Humana's website at 
https://www.humana.com/careers/. You can use their search

In [36]:
pred = uncompiled_fs("Is there education requirements or certifications for this position?",example1)

In [37]:
print(pred.answer)

Yes, a Bachelor's degree and relevant work experience are required; certification might also be implied based on 
industry standards.

In [38]:
llm.inspect_history(n=1)




Answer questions with short factoid answers.

---

Follow the following format.

Context: may contain relevant facts

Question: ${question}

Reasoning: Let's think step by step in order to ${produce the answer}. We ...

Answer: often between 1 and 5 words

---

Context:

6 hours
Associate Actuary - SPA Rx
Cincinnati, OH 45217
**Description** The Associate Actuary, Pricing is responsible for setting pricing assumptions, submitting bids, filing and gaining approval of premium rates and rate certifications with regulatory agencies. Supports implementation of rates, new plans and benefit changes. Provides guidance to Product Development on new product/benefit cost impacts. Develops and maintains pricing tools that price standard and custom benefits. The Associate Actuary, Pricing work assignments involve moderately complex to complex issues where the analysis of situations or data requires an in-depth evaluation of variable factors. **Responsibilities** _This a remote nationwide positio

"\n\n\nAnswer questions with short factoid answers.\n\n---\n\nFollow the following format.\n\nContext: may contain relevant facts\n\nQuestion: ${question}\n\nReasoning: Let's think step by step in order to ${produce the answer}. We ...\n\nAnswer: often between 1 and 5 words\n\n---\n\nContext:\n\n6 hours\nAssociate Actuary - SPA Rx\nCincinnati, OH 45217\n**Description** The Associate Actuary, Pricing is responsible for setting pricing assumptions, submitting bids, filing and gaining approval of premium rates and rate certifications with regulatory agencies. Supports implementation of rates, new plans and benefit changes. Provides guidance to Product Development on new product/benefit cost impacts. Develops and maintains pricing tools that price standard and custom benefits. The Associate Actuary, Pricing work assignments involve moderately complex to complex issues where the analysis of situations or data requires an in-depth evaluation of variable factors. **Responsibilities** _This a 